In [1]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import col

In [2]:
S3_BUCKET = "datalake-teste2"
S3_BRONZE = f"s3a://{S3_BUCKET}/bronze"
S3_SILVER = f"s3a://{S3_BUCKET}/silver"
S3_GOLD = f"s3a://{S3_BUCKET}/gold"

SPARK_MASTER = "local[*]"
SPARK_APP_NAME = "medallion-pipeline"

print(f"Bronze:  {S3_BRONZE}")
print(f"Silver:  {S3_SILVER}")
print(f"Gold:    {S3_GOLD}")

Bronze:  s3a://datalake-teste2/bronze
Silver:  s3a://datalake-teste2/silver
Gold:    s3a://datalake-teste2/gold


In [3]:
spark = SparkSession.builder \
    .appName(SPARK_APP_NAME) \
    .master(SPARK_MASTER) \
    .config("spark.hadoop.fs.s3a.endpoint", "http://localhost:4566") \
    .config("spark.hadoop.fs.s3a.access.key", "test") \
    .config("spark.hadoop.fs.s3a.secret.key", "test") \
    .config("spark.hadoop.fs.s3a.path.style.access", "true") \
    .config("spark.hadoop.fs.s3a.connection.ssl.enabled", "false") \
    .config("spark.hadoop.fs.s3a.impl", "org.apache.hadoop.fs.s3a.S3AFileSystem") \
    .config("spark.jars.packages", "org.apache.hadoop:hadoop-aws:3.3.2,com.amazonaws:aws-java-sdk-bundle:1.12.261") \
    .getOrCreate()

26/09/02 22:58:04 WARN Utils: Your hostname, hugo resolves to a loopback address: 127.0.1.1; using 10.159.141.203 instead (on interface wlp2s0)
26/09/02 22:58:04 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address


:: loading settings :: url = jar:file:/home/hugo/Desktop/Project/venv/lib/python3.13/site-packages/pyspark/jars/ivy-2.5.1.jar!/org/apache/ivy/core/settings/ivysettings.xml


Ivy Default Cache set to: /home/hugo/.ivy2/cache
The jars for the packages stored in: /home/hugo/.ivy2/jars
org.apache.hadoop#hadoop-aws added as a dependency
com.amazonaws#aws-java-sdk-bundle added as a dependency
:: resolving dependencies :: org.apache.spark#spark-submit-parent-d1945c6f-b656-4e64-8629-eee17a00446c;1.0
	confs: [default]
	found org.apache.hadoop#hadoop-aws;3.3.2 in central
	found org.wildfly.openssl#wildfly-openssl;1.0.7.Final in central
	found com.amazonaws#aws-java-sdk-bundle;1.12.261 in central
:: resolution report :: resolve 323ms :: artifacts dl 12ms
	:: modules in use:
	com.amazonaws#aws-java-sdk-bundle;1.12.261 from central in [default]
	org.apache.hadoop#hadoop-aws;3.3.2 from central in [default]
	org.wildfly.openssl#wildfly-openssl;1.0.7.Final from central in [default]
	:: evicted modules:
	com.amazonaws#aws-java-sdk-bundle;1.11.1026 by [com.amazonaws#aws-java-sdk-bundle;1.12.261] in [default]
	------------------------------------------------------------------

In [4]:
import sys

from pyspark.sql import SparkSession, Window, functions as F

# Ordem estável das colunas na saída: chaves primeiro, depois um bloco por
# fonte, na mesma ordem de FONTES.
COLUNAS = [
    "transaction_datetime", "transaction_date", "purchase_id", "origem_evento",
    "buyer_id", "prod_item_id", "order_date", "release_date", "producer_id",
    "product_id", "item_quantity", "purchase_value",
    "subsidiary",
]

O qe está sendo feito:
- Está lendo em todas as 3 tabelas da camada Bronze, e empilhando elas em um dataset só

In [5]:
df_eventos = None
# Fontes de eventos (CDC) que alimentam a tabela final
FONTES = ["purchase", "product_item", "purchase_extra_info"]

for fonte in FONTES:

    df = spark.read.parquet(f"{S3_BRONZE}/{fonte}")

    # origem_evento não é só rastreabilidade. É ela que vai distinguir
    # "essa fonte não fala sobre a coluna" (herda o valor anterior) de
    # "essa fonte falou e o valor é vazio" (não herda) — a diferença que
    # o forward fill precisa respeitar para não ignorar cancelamentos.
    df = df.withColumn("origem_evento", F.lit(fonte))
    
    # unionByName casa por nome e preenche com NULL as colunas ausentes.
    # union() casaria por posição e erraria em silêncio se a ordem das
    # colunas divergisse entre as fontes.
    df_eventos = df if df_eventos is None else df_eventos.unionByName(
            df, allowMissingColumns=True
        )

df_eventos = df_eventos.select(*COLUNAS)

26/09/02 22:58:10 WARN MetricsConfig: Cannot locate configuration: tried hadoop-metrics2-s3a-file-system.properties,hadoop-metrics2.properties


In [6]:
df_eventos.show()

+--------------------+----------------+-----------+-------------+--------+------------+----------+------------+-----------+----------+-------------+--------------+----------+
|transaction_datetime|transaction_date|purchase_id|origem_evento|buyer_id|prod_item_id|order_date|release_date|producer_id|product_id|item_quantity|purchase_value|subsidiary|
+--------------------+----------------+-----------+-------------+--------+------------+----------+------------+-----------+----------+-------------+--------------+----------+
| 2023-05-22 07:20:00|      2023-05-22|         74|     purchase|  667788|          66|2023-05-21|  2023-05-22|     963963|      NULL|         NULL|          NULL|      NULL|
| 2023-05-22 07:20:00|      2023-05-22|         74|     purchase|  667788|          66|2023-05-21|  2023-05-22|     963963|      NULL|         NULL|          NULL|      NULL|
| 2023-06-25 12:00:00|      2023-06-25|         76|     purchase|  889900|          88|2023-06-24|  2023-06-25|     963963|  

In [7]:
conteudo = [
        F.coalesce(F.col(c).cast("string"), F.lit("<null>"))
        for c in sorted(COLUNAS)
    ]
df_eventos = df_eventos.withColumn(
        "hash_evento", F.sha2(F.concat_ws("||", *conteudo), 256))

In [8]:
df_eventos.show()

+--------------------+----------------+-----------+-------------+--------+------------+----------+------------+-----------+----------+-------------+--------------+----------+--------------------+
|transaction_datetime|transaction_date|purchase_id|origem_evento|buyer_id|prod_item_id|order_date|release_date|producer_id|product_id|item_quantity|purchase_value|subsidiary|         hash_evento|
+--------------------+----------------+-----------+-------------+--------+------------+----------+------------+-----------+----------+-------------+--------------+----------+--------------------+
| 2023-05-22 07:20:00|      2023-05-22|         74|     purchase|  667788|          66|2023-05-21|  2023-05-22|     963963|      NULL|         NULL|          NULL|      NULL|80ac8f42af1b27d0c...|
| 2023-05-22 07:20:00|      2023-05-22|         74|     purchase|  667788|          66|2023-05-21|  2023-05-22|     963963|      NULL|         NULL|          NULL|      NULL|80ac8f42af1b27d0c...|
| 2023-06-25 12:00:0

In [9]:
df_eventos = df_eventos.dropDuplicates(["hash_evento"])

O que está sendo feito:
- Pegando sempre o último evento daquele dia com base no purchase_id e transaction_datetime

In [10]:
ultimo_do_dia = Window.partitionBy("purchase_id", "origem_evento", "transaction_date").orderBy(
        F.col("transaction_datetime").desc(), F.col("hash_evento").desc(),)

O que está sendo feito:
- Filtrando somente a última transação daquele dia

In [11]:
df_eventos = (
    df_eventos.withColumn("_rn", F.row_number().over(ultimo_do_dia))
    .filter(F.col("_rn") == 1)
    .drop("_rn")
)

df_eventos.show(100)

26/09/02 22:58:22 WARN SparkStringUtils: Truncated the string representation of a plan since it was too large. This behavior can be adjusted by setting 'spark.sql.debug.maxToStringFields'.


+--------------------+----------------+-----------+-------------------+--------+------------+----------+------------+-----------+----------+-------------+--------------+-------------+--------------------+
|transaction_datetime|transaction_date|purchase_id|      origem_evento|buyer_id|prod_item_id|order_date|release_date|producer_id|product_id|item_quantity|purchase_value|   subsidiary|         hash_evento|
+--------------------+----------------+-----------+-------------------+--------+------------+----------+------------+-----------+----------+-------------+--------------+-------------+--------------------+
| 2023-01-20 22:02:00|      2023-01-20|         55|       product_item|    NULL|        NULL|      NULL|        NULL|       NULL|    696969|           10|         50.00|         NULL|56bfb324c970e2e7b...|
| 2023-07-12 09:00:00|      2023-07-12|         55|       product_item|    NULL|        NULL|      NULL|        NULL|       NULL|    696969|           10|         55.00|         NU

In [12]:
df_eventos.filter(col('purchase_id') == 55).show()

+--------------------+----------------+-----------+-------------------+--------+------------+----------+------------+-----------+----------+-------------+--------------+----------+--------------------+
|transaction_datetime|transaction_date|purchase_id|      origem_evento|buyer_id|prod_item_id|order_date|release_date|producer_id|product_id|item_quantity|purchase_value|subsidiary|         hash_evento|
+--------------------+----------------+-----------+-------------------+--------+------------+----------+------------+-----------+----------+-------------+--------------+----------+--------------------+
| 2023-01-20 22:02:00|      2023-01-20|         55|       product_item|    NULL|        NULL|      NULL|        NULL|       NULL|    696969|           10|         50.00|      NULL|56bfb324c970e2e7b...|
| 2023-07-12 09:00:00|      2023-07-12|         55|       product_item|    NULL|        NULL|      NULL|        NULL|       NULL|    696969|           10|         55.00|      NULL|4fc1dc512d56

O ue está sendo feito:
- Apagando a coluna has_evento e ordernando o df por purchase_id, transaction_datetime, origem_evento

- Salvando a tabela na silver como eventos_unificados e particionados por transaction_date

In [36]:
df_eventos.drop("hash_evento").orderBy("purchase_id", "transaction_datetime", "origem_evento").show(50, truncate=False)

df_eventos.write.mode("overwrite")\
    .partitionBy("transaction_date")\
        .parquet(f"{S3_SILVER}/eventos_unificados")

print(f"💾 Salvo em {S3_SILVER}/eventos_unificados")
print("=" * 60)

+--------------------+----------------+-----------+-------------------+--------+------------+----------+------------+-----------+----------+-------------+--------------+-------------+
|transaction_datetime|transaction_date|purchase_id|origem_evento      |buyer_id|prod_item_id|order_date|release_date|producer_id|product_id|item_quantity|purchase_value|subsidiary   |
+--------------------+----------------+-----------+-------------------+--------+------------+----------+------------+-----------+----------+-------------+--------------+-------------+
|2023-01-20 19:00:00 |2023-01-20      |55         |purchase           |15947   |5           |2023-01-20|2023-01-20  |852852     |NULL      |NULL         |NULL          |NULL         |
|2023-01-20 19:02:00 |2023-01-20      |55         |product_item       |NULL    |NULL        |NULL      |NULL        |NULL       |696969    |10           |50.00         |NULL         |
|2023-01-22 21:05:00 |2023-01-23      |55         |purchase_extra_info|NULL    |

💾 Salvo em s3a://datalake-teste2/silver/eventos_unificados
